# Step 01c — ERA5 BSISO-Domain Precipitation Download
**Project:** ENSO-BSISO Self-Supervised Learning  
**Author:** Jiayi (jh9141@nyu.edu)

This notebook downloads ERA5 daily total precipitation (`tp`) for use as a downstream
forecast-skill evaluation target (Plan 3).

**Variable:** `total_precipitation` — daily accumulated precipitation (units: m, metres of water equivalent)  
**Domain:** **60°E–160°E, 0–60°N, 2° resolution** — same grid as `u850`, `v850`, and OLR in notebooks 01 and 01b  
**Period:** May–September 1979–2023 (MJJAS, 45 years)  
**Time:** 12:00 UTC (consistent with all other fields in this project)

> **Why the full BSISO domain instead of the East Asian box only?**  
> Downloading over the same 60°E–160°E, 0–60°N grid means notebook 09 can produce
> *spatial skill maps* — ACC at every 2° grid point — showing where each representation
> has predictive power across the entire BSISO active domain.  
> The East Asian monsoon subregion (20–45°N, 100–145°E) is then extracted *in notebook 09*
> for the headline area-averaged skill score. One download, two analyses.

> **Note on `tp` at 12:00 UTC.** ERA5 `tp` is an accumulated quantity. The value at 12:00 UTC
> represents precipitation accumulated since the beginning of the short-range forecast initialized
> at 06:00 UTC (a ~6-hour window). For intraseasonal-timescale analysis (BSISO/ENSO), this
> single daily snapshot is a standard proxy. All downstream preprocessing (Lee et al. anomaly
> removal + normalization) is applied before regression.

**Output file:**
```
data/raw/precip_MJJAS_1979_2023.nc      (~40–60 MB, same size as OLR_MJJAS)
```

---
⚠️ **Prerequisite:** CDS API key must be set up (same as Notebooks 01 and 01b).  
⚠️ **Licence:** You already accepted the ERA5 single-levels licence for OLR in Session 4 — no new action needed.

## Cell 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

PROJECT_DIR = '/content/drive/MyDrive/BSISO_SSL_Project'
RAW_DIR     = f'{PROJECT_DIR}/data/raw'

os.makedirs(RAW_DIR, exist_ok=True)
print('Google Drive mounted.')
print(f'Raw data directory: {RAW_DIR}')

## Cell 2 — Install CDS API Client

In [ ]:
!pip install cdsapi --quiet
import cdsapi
print('cdsapi ready.')

## Cell 3 — Set Up CDS API Credentials

In [ ]:
# ============================================================
# FILL IN YOUR PERSONAL ACCESS TOKEN HERE
# Copy it from notebook 01b Cell 3 — same token
# ============================================================
CDS_API_KEY = 'YOUR_TOKEN_HERE'
# ============================================================

cdsapirc = f'url: https://cds.climate.copernicus.eu/api\nkey: {CDS_API_KEY}\n'
with open(os.path.expanduser('~/.cdsapirc'), 'w') as f:
    f.write(cdsapirc)

print('CDS credentials saved.')

try:
    client = cdsapi.Client(quiet=True)
    print('CDS API connection: OK')
except Exception as e:
    print(f'CDS API connection FAILED: {e}')

## Cell 4 — Download Total Precipitation (MJJAS, all years, single request)

Same domain as the atmospheric fields: **60°E–160°E, 0–60°N** → 31 lat × 51 lon grid.  
Single variable (no pressure level) → single request, similar size to OLR_MJJAS_1979_2023.nc.  
**Estimated time:** 10–25 min (CDS queue dependent)  
**Estimated file size:** ~40–60 MB

In [ ]:
import cdsapi
import os

if not os.path.exists(os.path.expanduser('~/.cdsapirc')):
    raise RuntimeError('Run Cell 3 first to set up CDS credentials.')

client = cdsapi.Client()

MJJAS_MONTHS = ['05', '06', '07', '08', '09']
DAYS         = [f'{d:02d}' for d in range(1, 32)]  # CDS ignores invalid dates
ALL_YEARS    = [str(y) for y in range(1979, 2024)]

# Same domain as atmospheric fields — CDS area format: [N, W, S, E]
BSISO_AREA = [60, 60, 0, 160]

out_precip = f'{RAW_DIR}/precip_MJJAS_1979_2023.nc'

if os.path.exists(out_precip):
    size_mb = os.path.getsize(out_precip) / 1e6
    print(f'[SKIP] precip_MJJAS_1979_2023.nc already exists ({size_mb:.1f} MB)')
else:
    print('Downloading ERA5 total precipitation (MJJAS 1979–2023) ...')
    print(f'Domain: {BSISO_AREA} [N, W, S, E]  →  60°E–160°E, 0–60°N')
    print('Do NOT close the tab.')

    client.retrieve(
        'reanalysis-era5-single-levels',
        {
            'product_type': 'reanalysis',
            'variable'    : 'total_precipitation',
            'year'        : ALL_YEARS,
            'month'       : MJJAS_MONTHS,
            'day'         : DAYS,
            'time'        : '12:00',
            'area'        : BSISO_AREA,
            'grid'        : [2.0, 2.0],
            'data_format' : 'netcdf',
        },
        out_precip
    )

    size_mb = os.path.getsize(out_precip) / 1e6
    print(f'Saved: precip_MJJAS_1979_2023.nc  ({size_mb:.1f} MB)')

## Cell 5 — Verify Download

In [ ]:
import xarray as xr
import numpy as np
import pandas as pd

print('=' * 60)
print('VERIFICATION REPORT — ERA5 Precipitation Download')
print('=' * 60)

ds = xr.open_dataset(out_precip)
print(f'\nDataset variables: {list(ds.data_vars)}')
print(f'Dataset dims:      {dict(ds.dims)}')

# Identify time dimension name (new CDS API uses 'valid_time')
time_dim = 'valid_time' if 'valid_time' in ds.dims else 'time'
times = pd.DatetimeIndex(ds[time_dim].values)

print(f'\n[1] Time coverage')
print(f'  N days:        {len(times)}')
print(f'  First:         {str(times[0])[:10]}')
print(f'  Last:          {str(times[-1])[:10]}')
months_found = sorted(set(times.month))
print(f'  Months found:  {months_found}  (expected [5, 6, 7, 8, 9])')
years_found = sorted(set(times.year))
print(f'  Years found:   {years_found[0]}–{years_found[-1]}  ({len(years_found)} years)')

print(f'\n[2] Grid')
print(f'  Latitude:  {float(ds.latitude.min()):.0f}–{float(ds.latitude.max()):.0f}°N  ({len(ds.latitude)} points, expected 31)')
print(f'  Longitude: {float(ds.longitude.min()):.0f}–{float(ds.longitude.max()):.0f}°E  ({len(ds.longitude)} points, expected 51)')
print(f'  Same grid as u850/v850/OLR: {len(ds.latitude) == 31 and len(ds.longitude) == 51}')

tp_var = 'tp' if 'tp' in ds.data_vars else list(ds.data_vars)[0]
tp = ds[tp_var].values

print(f'\n[3] Precipitation values')
print(f'  Variable name: "{tp_var}"')
print(f'  Units (from attr): {ds[tp_var].attrs.get("units", "not set")}')
print(f'  Min: {tp.min():.6f} m')
print(f'  Max: {tp.max():.6f} m')
print(f'  Mean: {tp.mean():.6f} m')

n_neg = int((tp < -1e-6).sum())  # ignore tiny rounding negatives
n_nan = int(np.isnan(tp).sum())
print(f'\n[4] Quality checks')
print(f'  Negative values (< -1e-6): {n_neg}  (expected 0; tiny ~−1e-9 values are ERA5 artefacts)')
print(f'  NaN values: {n_nan}  (expected 0)')

size_mb = os.path.getsize(out_precip) / 1e6
print(f'\n[5] File size: {size_mb:.1f} MB')

print('\nVerification complete.')
ds.close()

## Cell 6 — Quick Plot (Visual Sanity Check)

Compare climatological monthly mean precipitation for May, July, and September.
Physically expected:
- **July:** peak monsoon — heavy rainfall over Bay of Bengal, India, eastern China, western Pacific
- **May:** pre-monsoon — lighter, shifted south
- **September:** retreating monsoon — rainfall shifting southward

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

ds = xr.open_dataset(out_precip)
time_dim = 'valid_time' if 'valid_time' in ds.dims else 'time'
tp_var   = 'tp' if 'tp' in ds.data_vars else list(ds.data_vars)[0]

times = pd.DatetimeIndex(ds[time_dim].values)
tp    = np.clip(ds[tp_var].values, 0, None)  # clip tiny negatives

def monthly_mean(month):
    return tp[times.month == month].mean(axis=0)

tp_may  = monthly_mean(5)
tp_july = monthly_mean(7)
tp_sep  = monthly_mean(9)

lats = ds.latitude.values
lons = ds.longitude.values
extent = [lons.min(), lons.max(), lats.min(), lats.max()]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('ERA5 Total Precipitation climatology (12:00 UTC, 1979–2023)\n60°E–160°E, 0–60°N — same domain as atmospheric fields',
             fontsize=12, fontweight='bold')

titles = ['May mean', 'July mean (peak monsoon)', 'September mean']
data   = [tp_may, tp_july, tp_sep]
vmax   = max(d.max() for d in data)

for ax, d, title in zip(axes, data, titles):
    im = ax.imshow(d, cmap='Blues', origin='upper',
                   extent=extent, aspect='auto', vmin=0, vmax=vmax)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Longitude (°E)')
    ax.set_ylabel('Latitude (°N)')
    # Mark East Asian subregion box (20-45N, 100-145E)
    from matplotlib.patches import Rectangle
    rect = Rectangle((100, 20), 45, 25, linewidth=2, edgecolor='red',
                      facecolor='none', linestyle='--', label='EA box')
    ax.add_patch(rect)
    plt.colorbar(im, ax=ax, label='tp (m)', shrink=0.75)

axes[0].legend(loc='upper right', fontsize=8)
plt.tight_layout()
plt.show()

print('Red dashed box = East Asian subregion (20–45°N, 100–145°E) used for headline skill score in nb 09.')
ds.close()

## Cell 7 — Units and Domain Notes

Quick check of domain-averaged magnitudes per month, and a summary of how
this file feeds into notebook 09.

In [ ]:
import xarray as xr
import numpy as np
import pandas as pd

ds = xr.open_dataset(out_precip)
time_dim = 'valid_time' if 'valid_time' in ds.dims else 'time'
tp_var   = 'tp' if 'tp' in ds.data_vars else list(ds.data_vars)[0]

times  = pd.DatetimeIndex(ds[time_dim].values)
tp_m   = np.clip(ds[tp_var].values, 0, None)
tp_mm  = tp_m * 1000  # metres → mm

# Full-domain area average
tp_domain_mm = tp_mm.mean(axis=(1, 2))

print('Full-domain area-averaged tp at 12:00 UTC (1979–2023 MJJAS):')
for m, name in zip([5,6,7,8,9], ['May','Jun','Jul','Aug','Sep']):
    mask = times.month == m
    v = tp_domain_mm[mask]
    print(f'  {name}: mean={v.mean():.3f} mm  max={v.max():.3f} mm  (N={mask.sum()} days)')

# East Asian subregion average
lats = ds.latitude.values
lons = ds.longitude.values
ea_lat = (lats >= 20) & (lats <= 45)
ea_lon = (lons >= 100) & (lons <= 145)
tp_ea_mm = tp_mm[:, ea_lat, :][:, :, ea_lon].mean(axis=(1, 2))

print()
print('East Asian subregion (20–45°N, 100–145°E) area-averaged tp:')
for m, name in zip([5,6,7,8,9], ['May','Jun','Jul','Aug','Sep']):
    mask = times.month == m
    v = tp_ea_mm[mask]
    print(f'  {name}: mean={v.mean():.3f} mm  max={v.max():.3f} mm')

print()
print('Notebook 09 usage plan:')
print('  1. Apply Lee et al. preprocessing to tp over full domain (31 x 51)')
print('  2. Spatial skill map: ACC at each grid point across full BSISO domain')
print('  3. Headline skill score: area-average over EA subregion (20-45N, 100-145E)')
print('  4. Compare: BSISO index / supervised 2D / SSL 2D')

ds.close()

---
## Done!

If all cells ran successfully you should now have on Google Drive:

```
BSISO_SSL_Project/
└── data/
    └── raw/
        └── precip_MJJAS_1979_2023.nc    ← 31 lat × 51 lon, same grid as u850/v850/OLR
```

**Summary of what was downloaded:**
- Variable: `tp` (total precipitation, ERA5 single-levels)
- Domain: 60°E–160°E, 0–60°N → 31 lat × 51 lon = 1,581 grid points
- Period: May–September 1979–2023 (45 years, ~6,975 days)
- Time: 12:00 UTC daily
- Units: metres (m), ERA5 convention

**Next step:** Notebook `09_precip_forecast.ipynb` will:
1. Apply Lee et al. preprocessing to `tp` (annual cycle 3-harmonic Fourier + 120-day running mean + normalize)
2. Produce spatial skill maps — ACC at every 2° grid point across the full BSISO domain
3. Report headline area-averaged skill for the East Asian subregion (20–45°N, 100–145°E)
4. Compare all three representations: BSISO index / supervised 2D / SSL 2D

---
*DDCS Project | jh9141@nyu.edu*